# 02 · ML VaR Estimation — Quantile Regression vs Historical Simulation

**Project 8 / 10**  
XGBoost quantile regression for conditional 99% VaR.  
Kupiec POF backtesting. Basel III exception monitoring.

> Author: Niraj Neupane | github.com/nirajneupane17

In [ ]:
import sys; sys.path.insert(0,"../src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
from feature_engineering import build_var_features, train_test_split_temporal
from ml_models import MLVaREstimator
print("Libraries loaded")

In [ ]:
rets = pd.read_csv("../data/returns.csv", index_col="Date", parse_dates=True)
weights = np.array([0.4,0.2,0.2,0.1,0.1])
port = (rets * weights).sum(axis=1).dropna()
print(f"Portfolio daily vol: {port.std()*np.sqrt(252)*100:.1f}%")
print(f"99% Historical VaR : {np.percentile(port.values,1)*100:.3f}%")
port.describe()

In [ ]:
feat = build_var_features(port)
common = feat.index.intersection(port.index)
fc = feat.loc[common]; pc = port.loc[common]
sp = int(len(fc)*0.80)
X_tr=fc.drop(columns=["returns"]).iloc[:sp]; X_te=fc.drop(columns=["returns"]).iloc[sp:]
y_tr=pc.iloc[:sp]; y_te=pc.iloc[sp:]
var99_hist = np.percentile(y_tr.values,1)
print(f"Train: {X_tr.shape}  Test: {X_te.shape}")
print(f"Historical 99% VaR (train): {var99_hist*100:.3f}%")

In [ ]:
# Fit ML VaR estimator
var_est = MLVaREstimator(confidence_levels=(0.95, 0.99))
var_est.fit(X_tr, y_tr)
bt = var_est.backtest_all(X_te, y_te)
for cl, res in bt.items():
    print(f"{int(cl*100)}% VaR  exc_rate={res['exception_rate']:.3f}  "
          f"expected={res['expected_rate']:.3f}  Kupiec p={res['p_value']:.3f}  "
          f"{'PASS ✓' if res['pass'] else 'FAIL ✗'}")

In [ ]:
img = plt.imread("../results/02_var_ml_comparison.png")
fig, ax = plt.subplots(figsize=(15,8)); ax.imshow(img); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Exception rate vs Basel green/amber/red zones
exc_rate = bt[0.99]["exception_rate"]*100
print(f"Exception rate: {exc_rate:.2f}%")
print(f"Basel zone: {'Green (≤4 exc/250)' if exc_rate*2.5<=4 else 'Amber' if exc_rate*2.5<=9 else 'RED — model review required'}")